In [0]:
%run ./00_Project_Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
# Read Bronze Table
bronze_df = spark.read.format("delta").load(
    "/Volumes/log-analytics/bronze/key_volume/bronze_delta_v2"
)

print("Rows :", bronze_df.count())
print("Columns :", len(bronze_df.columns))

display(bronze_df.limit(20))

In [0]:
# Explore Data
bronze_df.printSchema()

In [0]:
display(
    bronze_df.groupBy("kafka_topic").count()
)

In [0]:
display(
    bronze_df.groupBy("log_level").count()
)

In [0]:
# remove duplicate events;
silver_df = bronze_df.dropDuplicates(["event_id"])

In [0]:
# validation 
print("Bronze :", bronze_df.count())
print("Silver :", silver_df.count())

In [0]:
# Converting string to timestamp
silver_df = silver_df.withColumn(
    "event_timestamp",
    F.to_timestamp("timestamp")
)

In [0]:
# Create Time Features
silver_df = (
    silver_df
    .withColumn("event_date", F.to_date("event_timestamp"))
    .withColumn("event_hour", F.hour("event_timestamp"))
    .withColumn("event_month", F.month("event_timestamp"))
    .withColumn("event_year", F.year("event_timestamp"))
    .withColumn("day_of_week", F.date_format("event_timestamp","EEEE"))
)
silver_df.printSchema()

In [0]:
# Severity Score
silver_df = silver_df.withColumn(
    "severity_score",
    F.when(F.col("log_level")=="INFO",1)
    .when(F.col("log_level")=="WARNING",2)
    .when(F.col("log_level")=="ERROR",3)
    .when(F.col("log_level")=="CRITICAL",4)
    .otherwise(0)
)

In [0]:
#Response Time Category
silver_df = silver_df.withColumn(
    "response_category",
    F.when(F.col("response_time") < 0.2, "Fast")
    .when(F.col("response_time") < 0.5, "Normal")
    .otherwise("Slow")
)

In [0]:
# Load Anomaly Labels
anomaly_df = (
    spark.read.text(
        "/Volumes/log-analytics/bronze/key_volume/anomaly_labels.txt"
    )
    .withColumnRenamed("value", "instance_id")
)
display(anomaly_df)

In [0]:
# removing the heading line and the blank one
anomaly_df = anomaly_df.filter(
    (~F.col("instance_id").startswith("The following")) &
    (F.trim(F.col("instance_id")) != "")
)
display(anomaly_df)

In [0]:
## Join labels
silver_df = (
    silver_df
    .join(
        anomaly_df.withColumn("is_anomaly", F.lit(1)),
        on = "instance_id",
        how = "left"
    )
)

In [0]:
## replace nulls
silver_df = silver_df.fillna(
    {"is_anomaly":0}
)

In [0]:
# Validate Labels
display(
    silver_df.groupBy("is_anomaly").count()
)

In [0]:
# Write Silver
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`log-analytics`.silver.silver_logs")
)

In [0]:
# Validation
silver = spark.table("`log-analytics`.silver.silver_logs")

print("Rows :", silver.count())
print("Columns :", len(silver.columns))

display(silver.limit(10))